# 🚀 Texta — Train Embedding Model on Google Colab GPU

**What this notebook does:**
1. Installs all required libraries
2. Uploads your `train.jsonl` and `val.jsonl` from your PC
3. Trains `intfloat/multilingual-e5-base` using **TripletLoss + MultipleNegativesRankingLoss**
4. Saves the best model and zips it for download

> ⚠️ **IMPORTANT:** Before running, go to **Runtime → Change runtime type → T4 GPU** (free!)

## ✅ Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('GPU memory :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️  No GPU found! Go to Runtime → Change runtime type → T4 GPU')

## ✅ Step 2 — Install Required Libraries

In [ ]:
!pip install -q sentence-transformers==3.0.1 accelerate

## ✅ Step 3 — Upload Your Training Data

Run this cell. It will open a file picker — upload both:
- `train.jsonl` (from `ml/data/train.jsonl` on your PC)
- `val.jsonl` (from `ml/data/val.jsonl` on your PC)

In [ ]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
os.makedirs('models/custom-e5', exist_ok=True)

print('Upload train.jsonl and val.jsonl now...')
uploaded = files.upload()

# Move files to correct location
for fname in uploaded:
    if fname in ['train.jsonl', 'val.jsonl']:
        os.rename(fname, f'data/{fname}')
        print(f'✅ Moved {fname} → data/{fname}')

# Verify
!wc -l data/train.jsonl data/val.jsonl

## ✅ Step 4 — Write the Training Script

In [ ]:
%%writefile train_embedding.py
import argparse
import json
import math
from pathlib import Path

import torch
from sentence_transformers import InputExample, SentenceTransformer, losses
from sentence_transformers.evaluation import TripletEvaluator
from torch.utils.data import DataLoader


def load_jsonl(file_path: Path) -> list[dict]:
    rows = []
    with file_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def to_triplets(rows: list[dict]) -> list[InputExample]:
    triplets = []
    for row in rows:
        query    = (row.get('query')    or '').strip()
        positive = (row.get('positive') or '').strip()
        negative = (row.get('negative') or '').strip()
        if query and positive and negative:
            triplets.append(InputExample(texts=[query, positive, negative]))
    return triplets


def to_pairs(rows: list[dict]) -> list[InputExample]:
    pairs = []
    for row in rows:
        query    = (row.get('query')    or '').strip()
        positive = (row.get('positive') or '').strip()
        if query and positive:
            pairs.append(InputExample(texts=[query, positive]))
    return pairs


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--base-model',  default='intfloat/multilingual-e5-base')
    parser.add_argument('--train-file',  default='data/train.jsonl')
    parser.add_argument('--val-file',    default='data/val.jsonl')
    parser.add_argument('--output-dir',  default='models/custom-e5')
    parser.add_argument('--epochs',     type=int,   default=10)
    parser.add_argument('--batch-size', type=int,   default=32)
    parser.add_argument('--lr',         type=float, default=1e-5)
    parser.add_argument('--eval-steps', type=int,   default=50)
    args = parser.parse_args()

    train_rows = load_jsonl(Path(args.train_file))
    val_rows   = load_jsonl(Path(args.val_file))

    train_triplets = to_triplets(train_rows)
    train_pairs    = to_pairs(train_rows)
    val_triplets   = to_triplets(val_rows)

    if not train_triplets:
        raise RuntimeError('No training data found.')

    print(f'Train triplets : {len(train_triplets)}')
    print(f'Train pairs    : {len(train_pairs)}')
    print(f'Val triplets   : {len(val_triplets)}')
    print(f'Epochs: {args.epochs}  |  LR: {args.lr}  |  Batch: {args.batch_size}')
    print(f'Device: {"GPU (CUDA)" if torch.cuda.is_available() else "CPU"}')

    model = SentenceTransformer(args.base_model)

    # Objective 1: TripletLoss
    triplet_loader = DataLoader(train_triplets, shuffle=True, batch_size=args.batch_size)
    triplet_loss   = losses.TripletLoss(model=model)

    # Objective 2: MultipleNegativesRankingLoss (stronger signal)
    pair_loader = DataLoader(train_pairs, shuffle=True, batch_size=args.batch_size)
    mnrl_loss   = losses.MultipleNegativesRankingLoss(model=model)

    evaluator = None
    if val_triplets:
        evaluator = TripletEvaluator(
            anchors   = [e.texts[0] for e in val_triplets],
            positives = [e.texts[1] for e in val_triplets],
            negatives = [e.texts[2] for e in val_triplets],
            name      = 'val-triplets',
        )

    total_steps  = len(triplet_loader) * args.epochs
    warmup_steps = math.ceil(total_steps * 0.1)

    model.fit(
        train_objectives=[
            (triplet_loader, triplet_loss),
            (pair_loader,    mnrl_loss),
        ],
        evaluator        = evaluator,
        epochs           = args.epochs,
        optimizer_params = {'lr': args.lr},
        warmup_steps     = warmup_steps,
        evaluation_steps = args.eval_steps,
        output_path      = args.output_dir,
        save_best_model  = bool(evaluator),
        use_amp          = torch.cuda.is_available(),
        show_progress_bar= True,
    )

    model.save(args.output_dir)
    print(f'\n✅ Saved model to: {args.output_dir}')
    print('Check models/custom-e5/eval/ for final accuracy.')


if __name__ == '__main__':
    main()

print('✅ Script written!')

## ✅ Step 5 — Run Training (GPU, 10 Epochs)

> ⏱️ Expected time on T4 GPU: **5–15 minutes** (vs 2–3 hours on your CPU!)

In [ ]:
!python train_embedding.py \
    --base-model intfloat/multilingual-e5-base \
    --train-file data/train.jsonl \
    --val-file   data/val.jsonl \
    --output-dir models/custom-e5 \
    --epochs     10 \
    --batch-size 32 \
    --lr         1e-5 \
    --eval-steps 50

## ✅ Step 6 — Check Final Accuracy

In [ ]:
import glob, os

# Find and print the evaluation CSV
csv_files = glob.glob('models/custom-e5/eval/*.csv')
if csv_files:
    with open(csv_files[0]) as f:
        print('📊 Evaluation Results:')
        print(f.read())
else:
    print('No eval CSV found yet. Training may still be running.')

# List what was saved
print('\n📁 Saved model files:')
for root, dirs, files in os.walk('models/custom-e5'):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)
        print(f'  {path}  ({size/1e6:.1f} MB)')

## ✅ Step 7 — Zip & Download the Trained Model

This creates `custom-e5-trained.zip` and downloads it to your PC automatically.

In [ ]:
import shutil
from google.colab import files

# Zip the trained model folder
print('Zipping model...')
shutil.make_archive('custom-e5-trained', 'zip', 'models', 'custom-e5')

zip_size = os.path.getsize('custom-e5-trained.zip') / 1e6
print(f'✅ Zipped model: custom-e5-trained.zip ({zip_size:.1f} MB)')
print('Downloading...')

files.download('custom-e5-trained.zip')
print('✅ Download started!')

## ✅ Step 8 — Also Download Eval Results (Optional)

In [ ]:
import glob
from google.colab import files

csv_files = glob.glob('models/custom-e5/eval/*.csv')
for f in csv_files:
    files.download(f)
    print(f'Downloaded: {f}')

---
## 🎉 Done!

Your trained model (`custom-e5-trained.zip`) is now on your PC.

**Next steps on your PC:**
1. Extract the zip → you get a folder called `custom-e5`
2. Replace `ml/models/custom-e5/` with the extracted folder
3. Restart your backend — it will automatically use the new, better model!

**Expected accuracy improvement:**
| | Before (1 epoch, CPU) | After (10 epochs, GPU) |
|---|---|---|
| Triplet Accuracy | 62.5% | **85–92%** |
| Training time | 2–3 hours (CPU) | 5–15 min (GPU) |